In [1]:
import csv
import os
import re
from datetime import datetime, timezone

In [2]:
## check what info python received from the website

import requests
from bs4 import BeautifulSoup


url = "https://store.steampowered.com/charts/topselling/US"

response = requests.get(
    url, 
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)

print("Status:", response.status_code)
print("Final URL:", response.url)
print("Content type:", response.headers.get("Content-Type"))
print("HTML length:", len(response.text))

soup = BeautifulSoup(response.text, "html.parser")
page_text = soup.get_text("", strip=True)

for term in ["Top Sellers", "Counter-Strike 2"]:
    print(f"{term!r} in HTML:", term.lower() in response.text.lower())
    print(f"{term!r} in visible text:", term.lower() in page_text.lower())

Status: 200
Final URL: https://store.steampowered.com/charts/topselling/US
Content type: text/html; charset=utf-8
HTML length: 749967
'Top Sellers' in HTML: True
'Top Sellers' in visible text: True
'Counter-Strike 2' in HTML: True
'Counter-Strike 2' in visible text: True


In [3]:
## Extracting Steam game names
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from urllib.request import urlopen

soup = BeautifulSoup(urlopen(url), "html.parser")
games = []

for link in soup.find_all("a", href=True):
    href = link["href"]
    name = link.get_text(" ", strip=True)
    path = urlparse(href).path
    if "/app/" in path and name:
        games.append(name)
games = list(dict.fromkeys(games))  # Remove duplicates
print("Total games found:", len(games))
for i, game in enumerate(games[:20], start=1):
    print(i, game)

Total games found: 100
1 Planet Zoo 2
2 AION 2
3 Hood Warfare 2
4 Counter-Strike 2
5 Valor Mortis
6 STAR WARS: Galactic Racer™
7 Battlefield™ 6
8 Castlevania: Belmont's Curse
9 ARC Raiders
10 Dead by Daylight
11 Season 5 Battlefield Pro - Battlefield™ 6 and REDSEC
12 Call of Duty®: Modern Warfare® 4
13 Apex Legends™
14 Risk of Rain 2
15 PUBG: BATTLEGROUNDS
16 Overwatch®
17 Halo: The Master Chief Collection
18 Marvel Rivals
19 WARDOGS
20 Dota 2


In [4]:
## inspect HTML around one game
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")
# find a Steam application link with visible text
link = next(
    (
        a for a in soup.find_all("a", href=True)
        if "/app/" in a ["href"] and a.get_text(strip=True)
    ),
    None
)

if link:
    print("GAME:", link.get_text("", strip=True))
    ##inspect larger parent elements
    parent = link

    for level in range(1, 7):
        parent = parent.parent
        if parent is None:
            break

        print(f"\n--- Parent level {level} ---")
        print(parent.get_text(" ", strip=True)[:500])
        print(parent.name, parent.get("class"))


GAME: Planet Zoo 2

--- Parent level 1 ---
Planet Zoo 2
td ['iV08nNqP--8-']

--- Parent level 2 ---
1 Planet Zoo 2 $49.99 ▲ 87 2
tr ['_6-rISlJoUe0-']

--- Parent level 3 ---
1 Planet Zoo 2 $49.99 ▲ 87 2 2 AION 2 Free To Play ▼ 1 4 3 Hood Warfare 2 -10% $5.99 $5.39 New 1 4 Counter-Strike 2 Free To Play 0 740 5 Valor Mortis -15% $44.98 $38.23 New 1 6 STAR WARS: Galactic Racer™ $59.99 ▲ 7 2 7 Battlefield™ 6 -60% $69.99 $27.99 ▲ 50 63 8 Castlevania: Belmont's Curse $29.99 New 1 9 ARC Raiders -50% $39.99 $19.99 ▲ 62 2 10 Dead by Daylight $19.99 ▲ 21 492 11 Season 5 Battlefield Pro - Battlefield™ 6 and REDSEC $24.99 New 1 12 Call of Duty®: Modern Warfare® 4 $69.99 ▲ 72 2 
tbody None

--- Parent level 4 ---
Rank Price Change Weeks 1 Planet Zoo 2 $49.99 ▲ 87 2 2 AION 2 Free To Play ▼ 1 4 3 Hood Warfare 2 -10% $5.99 $5.39 New 1 4 Counter-Strike 2 Free To Play 0 740 5 Valor Mortis -15% $44.98 $38.23 New 1 6 STAR WARS: Galactic Racer™ $59.99 ▲ 7 2 7 Battlefield™ 6 -60% $69.99 $27.99 ▲ 50 63 8 Cas